In [ ]:
# this project is to obtain a matrix table where we can take a look at the specific currencies intended. 
# for a start lets try to do a 10 x 10 matrix for the G10 currencies. 
# for columns it will be the BUY leg
# for the rows it will be SELL leg. 
# g10 currencies are as follows:
# 1. USD
# 2. EUR
# 3. JPY 
# 4. GBP
# 5. AUD
# 6. NZD
# 7. CAD
# 8. CHF
# 9. NOK
# 10. SEK
# lets go. 

# Lets also create a preset of regional currency pairs. 
# If i recall in BBG, we have quite a number of predefined sets of currencies. 
# 1. is the EMEA --> Emerging Markets? Or was it Middle East?
# 2. Asian ccy --> these are the major currencies in Asia. --> SGD, MYR, CNY/CNH, etc. 
# 3. African currencies? 
# 4. European currencies? (outside of the G10)

# So a few things to do.
# 1. get the preset defined lists for the currencies mentioned above. --> done
# 2. holiday handling for each currencies for each country. (something interesting to ponder, if its EUR, does it mean that EUR currencies will always have a quote if different parts of the union dont have any holidays?) 
# 3. handling missing nan data if the inverse exists. (i.e. if NZDSEK does not exist but its inverse does (SEKNZD) then we should do a special exception handling to find the inverse pair and invert it to its intended missing pair.) --> done


In [ ]:
import numpy as np
import pandas as pd
import os
import yfinance as yf # source data
import re
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import plotly.graph_objects as go

In [ ]:
# lets create a simple matrix 
# 1. Below is the G10 currency pairs.
g10_ccy_list = ['USD', 'EUR', 'JPY', 'GBP', 'AUD', 'NZD', 'CAD', 'CHF', 'NOK', 'SEK']

# 2. EMEA ccy list
emea_ccy_list = ['USD', 'EUR', 'GBP', 'CHF', 'SEK', 'NOK', 'PLN', 'HUF', 'TRY', 'ZAR']

# 3. Asian ccy list 
asian_ccy_list = ['USD', 'EUR', 'JPY', 'AUD', 'CNY', 'KRW', 'SGD', 'TWD', 'INR', 'HKD']

# 4. LatAm ccy list.
latam_ccy_list = ['USD', 'EUR', 'CAD', 'MXN', 'BRL', 'CLP', 'COP', 'PEN', 'ARS', 'CLF']

# 5. combine these preset ccy lists into a dict. 
preset_ccy_dict = {'g10_ccy_list' : g10_ccy_list,
                   'emea_ccy_list' : emea_ccy_list,
                   'asian_ccy_list' : asian_ccy_list,
                   'latam_ccy_list' : latam_ccy_list,
                  }

print(f"preset_ccy_dict is: {preset_ccy_dict}")

# for each column for each row, you need to obtain the necessary buy/sell legs
# if col == row then leave it blank?

master_ccy_pairs_dict = {}
master_ccy_pairs_only_dict = {}
for k, v in preset_ccy_dict.items():
    print(f"k, v vals are: ({k},{v})")
    ccy_pairs_dict = {}
    ccy_pairs_only_dict = {}
    for col in preset_ccy_dict[k]: # we are running the first list of the key values in preset_ccy_dict[k] = g10 ccy list
        # print(col)
        medium_list = []
        medium_dict = {}
        for row in preset_ccy_dict[k]: # we are then rerunning the list again in the key values for the preset_ccy_dict[k] = g10_ccy_list
            # print(row)
            if col == row:
                key_ccy_pair = col + row
                val_ccy_pair = 1
            else:
                key_ccy_pair = col + row
                yf_ticker = col + row + "=X"
                val_ccy_pair = 1
                #print(f"key_ccy_pair is: {key_ccy_pair}")
                #fx_rate_df = yf.download(yf_ticker, period = '1mo')['Close']
                #print(f"length of fx_rate_df is: \n {len(fx_rate_df)}")
                #print(f"index fx_rate_df is:\n {fx_rate_df.loc[fx_rate_df.index.max()].values}")
                #val_ccy_pair = fx_rate_df.loc[fx_rate_df.index.max()].values
            medium_dict[key_ccy_pair] = val_ccy_pair
            medium_list.append(key_ccy_pair)
        ccy_pairs_only_dict[col] = medium_list
        ccy_pairs_dict[col] = medium_dict
    master_ccy_pairs_dict[k] = ccy_pairs_dict
    master_ccy_pairs_only_dict[k] = ccy_pairs_only_dict
#ccy_pairs_only_dict
#ccy_pairs_dict
master_ccy_pairs_dict
master_ccy_pairs_only_dict

In [ ]:
def data_manipulation_fx_rate(input_df):
    
    # we need to cater for the missing currency pairs. 
    #print(len(input_df.columns))
    #print(len(yf_fx_rate_list))
    
    # lets remove the weekends so that we can only find the data that is not nan. 
    medium_df = input_df.copy()
    medium_df = medium_df[medium_df.index.weekday < 5]
    # take the max index
    # display(medium_df.columns)
    
    medium_max_df = medium_df.loc[medium_df.index.max()]
    display(medium_max_df)
    
    clean_tickers_name = [re.sub(r"=X$", "", ticker) for ticker in medium_max_df.index.tolist()]
    # we need to also clean up the naming in the yahoo downloads for the fx pair tickers. 
    medium_df.columns = clean_tickers_name
    display(medium_df.columns)
    
    # lets get the ccy pairs that are null in values.
    medium_max_df.index  = clean_tickers_name
    missing_pairs = medium_max_df.loc[medium_max_df.isna()].index.tolist()
    print(f"missing_pairs is: {missing_pairs}")
    print(f"length of missing_pairs is: {len(missing_pairs)}")
    
    # ccy_pair_val = medium_max_df.loc[medium_max_df.index == 'BRLCAD']
    # ccy_pair_val
    
    # 1. first lets find out if the inverse ccy pair has value. if it does then we can just equal the missing pair with the inverse pair. 
    # missing_pairs[0]
    
    for missing_ccy_pair in missing_pairs:
        print(f"current missing_ccy_pair is: {missing_ccy_pair}")
        regex_str = r"^([A-Z]{3})([A-Z]{3})$"
        inv_pair = re.sub(regex_str, r"\2\1", missing_ccy_pair) 
        print(f"missing_pairs[0] is: \n {missing_ccy_pair}")
        print(f"inv_pair is: {inv_pair}")
        inv_pair_val = 1/ medium_max_df.loc[medium_max_df.index == (inv_pair)]
        print(type(inv_pair_val.values))
        print(inv_pair_val.values)
        
        # 2. if the inverse leg isnt there then we have to find out if the USD base currency pair exist for both legs. if they do then we can just do a simple linear algebra to get the missing pair ccy. 
        # we need to replace the missing pair ccy with USD base. so if BRLCAD = BRLUSD x USDCAD
        match_ccy_pair = re.match(regex_str, missing_ccy_pair)
        match_ccy_pair
        ccy1 = match_ccy_pair.group(1) # keep the first pair and replace the second with USD
        ccy2 = match_ccy_pair.group(2) # keep the second pair and replace the first with USD
        num_ccy_pair = f"{ccy1}USD"
        denom_ccy_pair = f"USD{ccy2}"
        #print(f"num_ccy_pair, denom_ccy_pair is: ({num_ccy_pair}, {denom_ccy_pair})")
        # now lets find the ccy pair values for these two numerator and denominator ccy pairs. 
        num_ccy_val = medium_max_df.loc[medium_max_df.index == num_ccy_pair]
        denom_ccy_val = medium_max_df.loc[medium_max_df.index == denom_ccy_pair]
        
        # print(f"num_ccy_val and denom_ccy_val is: \n ({num_ccy_val.values}, {denom_ccy_val.values})")
        # final_val = num_ccy_val.values * denom_ccy_val.values
        # print(f"final_val for {missing_pairs[0]} is: {final_val}")
        
        if not (inv_pair_val.empty) and not pd.isna(inv_pair_val.iloc[0]):
            print(f"Yes we found the inverse value of the target missing pair!")
            medium_df[missing_ccy_pair] = 1/ medium_df[inv_pair]
            # print(f"medium_df after inversing is: {display(medium_df)}")
        
        elif not (num_ccy_val.empty) and not pd.isna(num_ccy_val.iloc[0]) and (not (denom_ccy_val.empty) and not pd.isna(denom_ccy_val.iloc[0])):
            print(f"We are finding two separate pairs denominated in USD so that we can find the missing pair ccy.")
            medium_df[missing_ccy_pair] = medium_df[num_ccy_pair] * medium_df[denom_ccy_pair]    
            # print(f"medium_df after multiplying the two ccy pairs is: {display(medium_df)}")
        else:
            print(f"Nope, no inverse value as well. we need to do the other way.")
    
    # print(f"medium_df with the inverse logic applied is: \n {display(medium_df[missing_pairs[0]])}")
    # print(f"new computed fx rates for missing pairs is:\n {display(medium_df)}")
    
    #medium_nan_df = medium_df.copy()
    #medium_nan_df = medium_nan_df.loc[medium_nan_df.index.max()]
    #medium_nan_list = medium_nan_df.loc[medium_nan_df.isna()].index.tolist()
    
    #print(medium_nan_list)
    return medium_df

# export to excel
#file_name = 'medium_fx_rate.xlsx'
#with pd.ExcelWriter(file_name) as writer:
#    medium_df.to_excel(writer, sheet_name = 'fx_rates', index = True)
#print(f"file_name has been exported to: ({os.getcwd()}/{file_name})")


In [ ]:
# now that we have the key = columns, values = col/values pair.
# lets see if we can concat the data together. 
# we shall create separate tables for each currencies and then go from there. 
# user defined on which currency to view so that we can make the code more efficient in only taking the preset currencies first.
target_ccy_set = 'g10_ccy_list'

#master_ccy_pairs_dict
#master_ccy_pairs_only_dict

ccy_matrix_df = pd.DataFrame.from_dict(master_ccy_pairs_only_dict[target_ccy_set], orient = 'index', columns = preset_ccy_dict[target_ccy_set])
ccy_matrix_df = ccy_matrix_df.T.copy()
display(ccy_matrix_df)

# take the off diagonals. 
yf_fx_rate_list = []
for i in range(len(ccy_matrix_df)):
    for j in range(len(ccy_matrix_df)):
        if i == j:
            #print(f"ccy is the same: {ccy_matrix_df.iloc[i,j]}.")
            #print(f"we shall skip this ccy pair")
            continue
        else:
            ccy_fx_pair = ccy_matrix_df.iloc[i,j] + "=X"
            yf_fx_rate_list.append(ccy_fx_pair)
print(yf_fx_rate_list)
print(len(yf_fx_rate_list))

all_fx_rate_df = yf.download(yf_fx_rate_list, period = '1mo')['Close']
# rearrange the columns to fit the matrix above.. 
all_fx_rate_df = all_fx_rate_df[yf_fx_rate_list]
print(display(all_fx_rate_df))

# before we implement the pct_change, we need to compute the missing ccy pairs by using the inverse or using the USD denomination.
all_fx_rate_df = data_manipulation_fx_rate(all_fx_rate_df)

# lets get the change between today vs yesterday. 
all_fx_rate_chg_df = all_fx_rate_df.pct_change(1)
all_fx_rate_chg_df

In [ ]:
all_fx_rate_max_df = all_fx_rate_df.copy()
# lets filter out the weekends data.
all_fx_rate_max_df = all_fx_rate_max_df[all_fx_rate_max_df.index.weekday < 5]
print(f"all_fx_rate_max_df for working days only filter is as follows: \n {display(all_fx_rate_max_df)}")

# then we can obtain the LAST WORKING DAY from the dataframes
all_fx_rate_max_df = all_fx_rate_max_df.loc[all_fx_rate_max_df.index.max()]

all_fx_rate_chg_max_df = all_fx_rate_chg_df.copy()
all_fx_rate_chg_max_df = all_fx_rate_chg_max_df[all_fx_rate_chg_max_df.index.weekday < 5]
all_fx_rate_chg_max_df = all_fx_rate_chg_max_df.loc[all_fx_rate_chg_max_df.index.max()]

# remove the "=X" to make it cleaner.
clean_tickers_name = [re.sub(r"=X$", "", ticker) for ticker in all_fx_rate_max_df.index.tolist()]

#clean_tickers_name
all_fx_rate_max_df.index = clean_tickers_name
all_fx_rate_chg_max_df.index = clean_tickers_name

print(f"all_fx_rate_max_df is: \n {display(all_fx_rate_max_df)}")
print(f"ccy_matrix_df is: \n {display(ccy_matrix_df)}")

# series to dictionary
all_fx_rate_max_dict = all_fx_rate_max_df.to_dict()
print(f"all_fx_rate_max_dict is: \n {all_fx_rate_max_dict}")
all_fx_rate_chg_max_df = all_fx_rate_chg_max_df.to_dict()
all_fx_rate_max_dict

rate_matrix_df = ccy_matrix_df.map(lambda x: all_fx_rate_max_dict.get(x, None)).fillna(0)
rate_change_matrix_df =ccy_matrix_df.map(lambda x: all_fx_rate_chg_max_df.get(x, None)).fillna(0)
display(rate_matrix_df)
display(rate_change_matrix_df)

In [ ]:
# lets put the heatmap matrix now. 

# 1. rotate the graph. 
heatmap_color_data = rate_change_matrix_df
heatmap_text_data = rate_matrix_df

# 2. create the custom magnitude color buckets
# colors: [Dark Red, Light red, Neutral/White, Light Green, Dark Green]
colors = ['#b30000', '#ff9999', '#f0f0f0', '#99ff99', '#006600']
custom_cmap = mcolors.ListedColormap(colors)

# Define boundaries based on criteria (-0.5%, -0.05%, 0.05%, 0.5%)
bounds = [-1, -0.005, -0.0005, 0.0005, 0.005, 1]
custom_norm = mcolors.BoundaryNorm(bounds, custom_cmap.N)

plt.figure(figsize=(13,7))

# plot the heatmap
sns.heatmap(
    heatmap_color_data,
    annot = heatmap_text_data, # Show the percentage values in the cells. 
    fmt = ".3f",   # Format numbers to 2 decimal places
    cmap = custom_cmap,   # Use custom discrete colors
    norm = custom_norm,   # Enforces the magnitude color rules
    cbar_kws = {'label' : 'Magnitude Scale Buckets (%)'},   
    linewidths = 1.0,   # Add thin lines between cells for readability.
    
)

plt.title("FX (G10 ccy) Matrix: Actual Rates Colored by Daily Change Magnitude", fontsize = 14, pad = 15)
plt.xlabel("Base Currency (Top)", fontsize = 12)
plt.ylabel("Quote Currency (Side)", fontsize = 12)
plt.show()

In [ ]:
# lets compare and use plotly. I feel like this would create a better and more friendly interface. 

# 1. set the same data 
# We have heatmap_color_data as the pct change of the rates on a daily basis.
# we also have the actual fx rates --> heatmap_text_data

display(heatmap_color_data)
display(heatmap_text_data)

# formatted currency text matrix for displace. 
annot_matrix = heatmap_text_data.map(lambda x : f"{x:.4f}").values
print(f"annot_matrix is: {annot_matrix}") # why?

# define discrete color buckets mapping to the scale
# bounds: [-1.0, -0.005, -0.001, 0.001, 0.005, 1.0]
# Plotly expects a colorscale mapped from 0 to 1, normalized evenly:
discrete_colorscale = [
    [0.000, '#333333'], [0.125, '#333333'],    # Masked Diagonal (0.0 to 0.125) 
    [0.125, '#3a0202'], [0.250, '#3a0202'],    # Bloomberg Deep Red 
    [0.250, '#aa0000'], [0.375, '#aa0000'],    # Bloomberg Red
    [0.375, '#ff4d4d'], [0.500, '#ff4d4d'],    # Bloomeberg Bright Red
    [0.500, '#1a1a1a'], [0.625, '#1a1a1a'],    # Dark Charcoal Neutral
    [0.625, '#00cc00'], [0.750, '#00cc00'],    # Soft Bright Green
    [0.750, '#008800'], [0.875, '#008800'],    # Mid Green
    [0.875, '#013220'], [1.000, '#013220'],    # Bloomberg Forest Green
]
print(f"if you are seeing this then we have ran through code blocks 1 till 1.")
# 2. Map color data values directly into uniform numeric bins [-1 to 1]
# this ensures Plotly breaks up the discrete color steps cleanly.
def digitize_values(val):
    if val <= -0.025:    return 1 # Deep Red
    if val <= -0.005:    return 2 # Red 
    if val <= -0.0005:   return 3 # Bright Red
    if val < 0.0005:     return 4 # Charcoal Neutral
    if val <= 0.005:     return 5 # Bright Neon Green
    if val <= 0.025:     return 6 # Green
    else:                return 7 # Forest green

color_bins = heatmap_color_data.map(digitize_values).values.copy() # add copy() at the end to make the array writable!
print(f"color_bins is: {color_bins}")

# 2.5: Map color bins and force diagonal to hit index 0 (Grey Mask block)
for i, row in enumerate(heatmap_color_data.index):
    for j, col in enumerate(heatmap_color_data.columns):
        if row == col:
            color_bins[i][j] = 0

# 3. built the plotly matrix
print(f"if you are seeing this then we have ran through code blocks 1 till 2.")
fig = go.Figure( data = go.Heatmap(
    z = color_bins,
    x = heatmap_color_data.columns,
    y = heatmap_color_data.index,
    colorscale = discrete_colorscale,
    showscale = True,
    zmin = -0.5,
    zmax = 7.5, # make sure this is 7.5 so it catches the 0.0 mask coordinates
    xgap = 2, # Added horizontal separation gap
    ygap = 2, # added vertical separation gap
    # Interactive custom hover template. 
    hovertemplate = "Base Ccy: %{x}<br>Quote Ccy: %{y}<br>Rate: %{text}<extra></extra>",
    text = annot_matrix,
    colorbar = dict(
        orientation = "h", # Flips the scale horizontally
        y = -0.28, # positions it cleanly below the matrix x-axis labels
        x = 0.5, # centers it horizontally
        xanchor = "center", 
        thickness = 22, # sets block height to fit text comfortably
        len = 1.0, # matches the full width of the matrix grid
        title = dict(text  = "Magnitude Scale Buckets", font = dict(color = "#ff9900", size = 12, family = "Courier New"), 
                     side = "top"), # Amber Color Font Title
        showticklabels = True,
        tickvals = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0],
        ticktext = [
            "Diagonal Mask",
            "< -2.5%", 
            "-2.5% to -0.5%", 
            "-0.5% to -0.05%", 
            "Neutral", 
            "0.05% to 0.5%", 
            "0.5% to 2.5%", 
            "> 2.5%"
        ],
        tickfont = dict(
            color = "white",
        size = 12,
        family = "Courier New",
        weight = "bold"), # Amber colorbar labels
        ticklabelposition = "inside" # injects the text directly inside the color bar rectangles                 
    )
))
print(f"if you are seeing this then we have ran through code blocks 1 till 3.")


# 4. superimpose actual fx rates text inside cells 
for i, row in enumerate(heatmap_color_data.index):
    for j, col in enumerate(heatmap_color_data.columns):
        #print(f"i is {i}, and row is: {row}")
        #print(f"j is {j}, and col is: {col}")
        cell_bin = color_bins[i][j]
        #print(f"cell_bin now is: {cell_bin}")
        # Pure Black text on bright neon cells, bright white text on deep/ neutral cells
        if row == col:
            display_text = "" # completely blank out diagonal text
            text_color = "rgba(0,0,0,0)"
        else:
            display_text = annot_matrix[i][j]
            # neutral cells look best with slightly muted gray to maintain hierarchy 
            # while all active colored cells get pure crisp white text
            text_color = "#b0b0b0" if cell_bin == 4 else "white"
            # 4 is Neutral
#            if cell_bin == 4:
#                text_color = "#b0b0b0"
#            # 3 is bright Red, 5 is Bright Neon Green
#            elif cell_bin in [3,5]:
#                text_color = "black"
#            # Dark outer colors
#            else:
#                text_color = "white"

        fig.add_annotation(
            x = col, 
            y = row,
            text = display_text,
            showarrow = False,

            font = dict(
                size = 13, 
                color = text_color,
                family = "Courier New, monospace", # Terminal style font
                weight = "bold"
            )
        )


# 4.5. Create a dict for the preset currencies as the title for the graph plot.
graph_title_dict = {"g10_ccy_list" : "FX (G10 ccy) Matrix: Actual FX Rates Colored by Change Magnitude",
                   'emea_ccy_list' : "FX (EMEA ccy) Matrix: Actual FX Rates Colored by Change Magnitude",
                   'asian_ccy_list' : "FX (Asian ccy) Matrix: Actual FX Rates Colored by Change Magnitude",
                   'latam_ccy_list' : "FX (LatAm ccy) Matrix: Actual FX Rates Colored by Change Magnitude",
                  }

print(f"if you are seeing this then we have ran through code blocks 1 till 4.")
# 5. clean layout styling for dark theme
fig.update_layout(
    title = dict(
        text = graph_title_dict[target_ccy_set],
        x = 0.5,
        y = 0.95,
        font = dict(size=16, color = "#ff9900", family = "Courier New")
    ),
    xaxis = dict(
        title = dict(text = "Base Currency (Top)", font=dict(color = "#ff9900", family = "Courier New")),
        side = "top", 
        tickangle = 0,
        tickfont = dict(color = "#ff9900", family = "Courier New"),
        showgrid = False # This removes the vertical lines inside the cells
        ),
    yaxis = dict(
        title = dict(text = "Quote Currency (Side)", font=dict(color = "#ff9900", family = "Courier New")),
        autorange = "reversed",
        tickfont = dict(color = "#ff9900", family = "Courier New"),
        showgrid = False # This removes the horizontal lines inside the cells.
    ),
    width = 1150, # Stretched from 900 to 1150 to widen the box 
    height = 480, # slightly taller to balance proportions.
    margin = dict(t = 100, b = 100, r = 40),
    paper_bgcolor = "#000000", # Pure Terminal Black/ outer margins remain pitch black
    plot_bgcolor = "#2a2a2a",  # Grid lines between cells turn into dark gray borders
)
print(f"if you are seeing this then we have ran through the entire code block and it is showing the figure below.")
fig.show()
print(f"plotly graph should appear above!")